In [0]:
# Agent4: ML Work
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Load silver dataset
df = spark.sql("SELECT * FROM samplesuperstore.silverdata.orders").toPandas()
print("Dataset shape:", df.shape)
df.head()


In [0]:
df['ProfitRatio'] = df['Profit'] / df['Sales']
df['DiscountRate'] = df['Discount'] / df['Sales']


In [0]:
X = df[['Quantity','Discount','Profit']]
y = df['Sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [0]:
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)


In [0]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Predictions already computed as y_pred
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)   # Root Mean Squared Error
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)   # R² Score

print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R² Score: {r2:.2f}")



In [0]:
%sql
USE CATALOG samplesuperstore;

CREATE OR REPLACE FUNCTION silverdata.run_ml(dataset STRING)
RETURNS STRING
LANGUAGE PYTHON
AS $$
def run_ml(dataset: str) -> str:
    from pyspark.sql import SparkSession
    from pyspark.ml.feature import VectorAssembler
    from pyspark.ml.regression import LinearRegression

    spark = SparkSession.builder.getOrCreate()
    df = spark.table(dataset)

    assembler = VectorAssembler(inputCols=["Sales"], outputCol="features")
    data = assembler.transform(df.select("Sales", "Profit")).withColumnRenamed("Profit", "label")

    lr = LinearRegression(featuresCol="features", labelCol="label")
    model = lr.fit(data)

    return f"ML Model Coeff: {model.coefficients}, Intercept: {model.intercept}"
$$;


In [0]:
%sql
-- Step 1: Switch to the correct catalog
USE CATALOG samplesuperstore;

-- Step 2: Call the UC function with silverdata.orders table
SELECT silverdata.run_data_cleaning('silverdata.orders');

-- Step 3: Verify function registration (optional check)
SHOW FUNCTIONS IN silverdata;
